# Multi-class Penguin Logistic Regression (PyTorch)

In [1]:
import numpy as np
import torch

In [2]:
# read the variables: class label and features

def string2float(item):
    return float(item) if item != '' else None

def read_strings(filename, col):
    with open(filename) as f:
        lines = f.readlines()
    values = [line.strip().split(',')[col] for line in lines[1:]]
    return values

def read_floats(filename, col):
    with open(filename) as f:
        lines = f.readlines()
    values = [string2float(line.strip().split(',')[col]) for line in lines[1:]]
    return values

def read_data_and_label(filename):
    x1 = np.array(read_floats(filename, col=2))  # bill length
    x2 = np.array(read_floats(filename, col=3))  # bill depth
    x3 = np.array(read_floats(filename, col=4))  # flipper length
    x4 = np.array(read_floats(filename, col=5))  # body mass
    y = np.array(read_strings(filename, col=0))  # species

    y[y == 'Adelie'] = 0
    y[y == 'Chinstrap'] = 1
    y[y == 'Gentoo'] = 2
    y = np.array(y).astype(float)

    idx = (x1 != None) & (x2 != None) & (x3 != None) & (x4 != None) & (y != None)
    x1 = x1[idx]
    x2 = x2[idx]
    x3 = x3[idx]
    x4 = x4[idx]
    y = y[idx]

    X = np.stack((x1, x2, x3, x4), axis=1).astype(float)
    y = y.reshape((y.size, 1)).astype(float)

    return X, y

In [3]:
def min_max_normalize(X, min_val=None, max_val=None):
    X = np.array(X)
    if min_val is None:
        min_val = np.min(X, axis=0, keepdims=True)
    if max_val is None:
        max_val = np.max(X, axis=0, keepdims=True)
    range_val = max_val - min_val
    range_val = np.where(range_val == 0, 1, range_val)
    X_norm = (X - min_val) / range_val
    return X_norm, min_val, max_val

In [4]:
def z_score_normalize(X, mean=None, std=None):
    if mean is None:
        mean = np.mean(X, axis=0)
    if std is None:
        std = np.std(X, axis=0)
    std = np.where(std == 0, 1, std)
    X_norm = (X - mean) / std
    return X_norm, mean, std

In [5]:
def print_formatted_vector(vec):
    s = ' '.join(f"{x.item():.4f}" for x in vec)
    print(s)

In [6]:
class LogisticRegressionModel(torch.nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = torch.nn.Linear(in_dim, out_dim)

    def forward(self, x):
        x = self.linear(x)
        x = torch.sigmoid(x)
        return x

In [11]:
# training data

filename = 'data/palmer-penguins/palmer-penguins-train.txt'
X, y = read_data_and_label(filename)

X, X_mean, X_std = z_score_normalize(X)

X = torch.from_numpy(X).float()
y = torch.from_numpy(y).float()

In [12]:
# init training model

torch.manual_seed(0)

model = LogisticRegressionModel(in_dim=X.shape[1], out_dim=3)

print(model.linear.weight)
print(model.linear.bias)

Parameter containing:
tensor([[-0.0037,  0.2682, -0.4115, -0.3680],
        [-0.1926,  0.1341, -0.0099,  0.3964],
        [-0.0444,  0.1323, -0.1511, -0.0983]], requires_grad=True)
Parameter containing:
tensor([-0.4777, -0.3311, -0.2061], requires_grad=True)


In [13]:
criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=0.1, weight_decay=0.000)

In [17]:
# training

for epoch in range(100):
    optimizer.zero_grad()
    y_pred = model(X)

    loss = criterion(y_pred, y.long().flatten())
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print('epoch {}: loss {:.4f}'.format(epoch + 1, loss.item()))
        # print_formatted_vector(y[::20])
        # print_formatted_vector(y_pred[::20])

epoch 20: loss 0.9049
epoch 40: loss 0.8533
epoch 60: loss 0.8188
epoch 80: loss 0.7939
epoch 100: loss 0.7747


In [18]:
print(model.linear.weight)
print(model.linear.bias)

Parameter containing:
tensor([[-0.7385,  0.5309, -0.7990, -0.6749],
        [ 0.3357,  0.5101, -0.1969,  0.0948],
        [ 0.3265, -0.6470,  0.5902,  0.6459]], requires_grad=True)
Parameter containing:
tensor([-0.2390, -0.6181, -0.2154], requires_grad=True)


In [19]:
# test data

filename = 'data/palmer-penguins/palmer-penguins-test.txt'

X, y = read_data_and_label(filename)

X, _, _ = z_score_normalize(X, X_mean, X_std)

X = torch.from_numpy(X).float()
y = torch.from_numpy(y).float()

In [20]:
# evaluation

model.eval()

with torch.no_grad():
    y_pred = model(X)
    y_pred = torch.argmax(y_pred, dim=1)
    y_corr = y.flatten() == y_pred
    print(torch.mean(y_corr.to(torch.float)))

tensor(0.8882)
